[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hanenalmayouf/applied-ml-workshop/blob/main/labs_colab/day1/lab_1_manafeth_leakage.ipynb)

# 🧭 مختبر اليوم الأول — كشف التسريب وصياغة مشكلة مغادرة العملاء

**ورشة أسس تعلم الآلة التطبيقي — اليوم 1 من 5**

هذا الدفتر مبني على مواصفات مختبرات «منافذ» المعتمدة للدورة، ومُجهَّز للعمل مباشرة في **Google Colab** أو في Jupyter محليًا.

**كيف تفتحه في Colab:** من قائمة `File → Upload notebook` في Colab ارفع هذا الملف. إذا لم يجد الدفتر مجلد البيانات تلقائيًا، ستظهر لك خانة لرفع ملف حزمة البيانات (`manafeth_data_package.zip`) المرفق بجوار هذا الدفتر — ارفعه وسيُستكمل التحميل تلقائيًا.

> 📌 راجع `00_start_here.md` قبل البدء لمعرفة طريقة استخدام خلايا **فكّر أولًا** و**TODO** و**مساعدة** في هذه الدفاتر.

In [ ]:
from pathlib import Path
import pandas as pd

# 1) نبحث عن مجلد البيانات بجانب هذا الدفتر (يعمل محليًا وفي Colab إذا رفعت المجلد كاملًا)
DATA_CANDIDATES = [Path("manafeth_data_package"), Path("data"), Path("../data/raw"), Path("data/raw")]
DATA_DIR = next((p for p in DATA_CANDIDATES if p.exists()), None)

# 2) إذا لم نجد المجلد ونحن داخل Google Colab، نطلب من الطالب رفع حزمة البيانات (ملف zip)
if DATA_DIR is None:
    try:
        from google.colab import files
        import zipfile

        print("لم يتم العثور على مجلد البيانات محليًا.")
        print("ارفع ملف حزمة البيانات (.zip) الذي يرافق هذا المختبر ثم انتظر انتهاء الرفع...")
        uploaded = files.upload()
        for name in uploaded:
            if name.lower().endswith(".zip"):
                with zipfile.ZipFile(name) as z:
                    z.extractall(".")
        DATA_DIR = next((p for p in DATA_CANDIDATES if p.exists()), Path("manafeth_data_package"))
    except ImportError:
        DATA_DIR = Path("manafeth_data_package")
        print("تنبيه: لسنا داخل Google Colab ولم يوجد مجلد بيانات — ضع حزمة البيانات بجانب الدفتر.")

CUSTOMERS_PATH = DATA_DIR / "manafeth_customers.parquet"
ORDERS_PATH = DATA_DIR / "manafeth_orders.parquet"
VEHICLES_PATH = DATA_DIR / "markabat_listings_sample.csv"
SHIFTED_PATH = DATA_DIR / "shifted_month.parquet"

print("مجلد البيانات المستخدم:", DATA_DIR.resolve())
assert CUSTOMERS_PATH.exists(), "تعذّر العثور على manafeth_customers.parquet — تأكد من رفع حزمة البيانات كاملة."

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (7, 4)

## 🎯 هدف المختبر

تستكشف جدول العملاء الفعلي وتحوّل طلب العمل إلى مسألة تصنيف محددة: **هل سيتوقف العميل عن إجراء طلبات مكتملة خلال الثلاثين يومًا التالية لتاريخ اللقطة؟** كما تتعلم قاعدة الوقت التي تمنع إدخال معلومات من المستقبل إلى النموذج.

## السيناريو والبيانات

يحتوي `manafeth_customers.parquet` على **48,000 صف**؛ كل صف يمثل **عميلًا واحدًا عند تاريخ لقطة شهري**. الهدف هو `churned_30d`. القيمة `1` تعني أن العميل لم يجرِ طلبًا مكتملًا في الثلاثين يومًا التالية، والقيمة `0` تعني أنه استمر في الطلب.

| نوع العمود | أعمدة واقعية من الملف | القرار |
|---|---|---|
| هدف | `churned_30d` | يبقى في `y` فقط |
| معرّف | `customer_id` | يُستبعد من الخصائص |
| خصائص آمنة مبدئيًا | `city`, `city_tier`, `device`, `payment_method`, `tenure_months`, `orders_per_month`, `avg_basket_sar`, `days_since_last_order`, `distinct_categories`, `promo_usage_rate`, `avg_rating`, `last_promo_used` | مرشحة لتدخل `X` بعد الفحص |
| **تسريب معلومات** | `refund_issued`, `support_ticket_after_snapshot`, `next_month_orders` | تُستبعد قطعًا — تحدث بعد اللقطة |
| تواريخ | `signup_date`, `snapshot_date` | لا تُستخدم كخصائص في هذا المختبر |

## 🤔 فكّر أولًا

قبل تشغيل أي كود: اقرأ وصف عمود `next_month_orders`. هل تتوفر قيمته لحظة اتخاذ قرار المتابعة مع العميل؟ لماذا لا يصلح كخاصية رغم أنه قد يرفع دقة النموذج ظاهريًا؟

In [ ]:
customers = pd.read_parquet(CUSTOMERS_PATH)
customers.head()

In [ ]:
customers.info()
customers.isna().sum().sort_values(ascending=False)

In [ ]:
customers["churned_30d"].value_counts(normalize=True)

### 📊 رسم: توزيع الهدف

مثال مُنفَّذ — لاحظ أن فئة `churned_30d = 1` أقل تكرارًا (بيانات غير متوازنة)، وهذا سيهمّنا في اليوم الرابع.

In [ ]:
counts = customers["churned_30d"].value_counts().sort_index()
plt.bar(["استمر (0)", "غادر (1)"], counts.values, color=["#33CC99", "#F26522"])
plt.title("توزيع هدف مغادرة العملاء خلال 30 يومًا")
plt.ylabel("عدد العملاء")
plt.show()

### 💡 مساعدة

إذا أردت رؤية أي الأعمدة فيها قيم ناقصة بصريًا لا رقميًا فقط، ارسم عدد القيم الناقصة لكل عمود كمخطط أعمدة أفقي.

In [ ]:
# TODO: احسب عدد القيم الناقصة لكل عمود من customers، واحتفظ فقط بالأعمدة التي فيها نقص (> 0)
missing = ...  # اكتب هنا

# TODO: ارسمها كمخطط أعمدة أفقي (kind="barh") بعنوان مناسب

## 📝 التصنيف: صنّف كل عمود

اختبر كل عمود مشكوك فيه بالسؤال: **«هل كانت هذه القيمة موجودة عند تاريخ اللقطة قبل بدء الثلاثين يومًا التالية؟»**. ثم أكمل القائمتين التاليتين.

In [ ]:
# TODO: أكمل قائمة الخصائص التي يمكن استخدامها بأمان (متاحة قبل لحظة القرار)
safe_features = [
    # اكتب أسماء الأعمدة الآمنة هنا، كنص بين علامتي اقتباس، مفصولة بفواصل
]

# TODO: أكمل قائمة الأعمدة التي يجب استبعادها لأنها تكشف معلومة من المستقبل
leak_columns = [
    # اكتب أسماء أعمدة التسريب الثلاثة هنا
]

print("خصائص آمنة:", safe_features)
print("أعمدة مسرّبة مستبعدة:", leak_columns)

## 🧠 صياغة المشكلة

اكتب في الخلية التالية (كنص Markdown) جملة بصيغة: «الصف يمثل …، والهدف هو …، والقرار الذي يساعده النموذج هو …».

_اكتب صياغتك هنا._

## النتيجة المتوقعة

يجب أن تلاحظ أن `avg_rating` و`last_promo_used` يحتويان على قيم ناقصة، وأن فئة المغادرة أقل من فئة الاستمرار. الأهم أن تسجّل قرارًا واضحًا باستبعاد الأعمدة الثلاثة المسرّبة، لا أن تكتفي بالقول إنها «أعمدة غير مناسبة».

## المهارات التي راجعتها

صياغة مشكلة عمل، تحديد وحدة التحليل والهدف والخصائص، قراءة جدول Parquet، فحص أنواع الأعمدة والنقص، وتطبيق اختبار الوقت لاكتشاف تسريب البيانات.

## ✅ تحقق ذاتيًا قبل إغلاق الدفتر
- [ ] `safe_features` لا يحتوي على `customer_id` ولا على أي عمود تاريخ
- [ ] `leak_columns` يحتوي على الأعمدة الثلاثة بالضبط: `refund_issued`, `support_ticket_after_snapshot`, `next_month_orders`
- [ ] كتبت صياغة المشكلة بجملة واحدة واضحة
- [ ] لاحظت عدم توازن فئة الهدف من الرسم البياني

**غدًا:** نبني خط معالجة (Pipeline) يعالج القيم الناقصة والأعمدة العددية والفئوية باستخدام `safe_features` نفسها.